In [1]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [2]:
import yfinance as yf
import pandas as pd

def fetch_stock_data(tickers, start, end):
    df = yf.download(tickers, start=start, end=end, auto_adjust=True)
    if df.empty:
        raise ValueError("No stock data found")

    if isinstance(df.columns, pd.MultiIndex):
        df = df.xs("Close", axis=1, level=0)
    elif "Close" in df.columns:
        df = df[["Close"]]

    return df


In [3]:
from crewai import Agent

market_data_agent = Agent(
    role="Market Data Agent",
    goal="Fetch and preprocess stock price data",
    backstory="Expert in financial market data pipelines",
    allow_delegation=False
)


In [4]:
stock_analyst_agent = Agent(
    role="Stock Analyst",
    goal="Analyze price trends and performance",
    backstory="Experienced equity research analyst",
    verbose=True
)


In [5]:
risk_analyst_agent = Agent(
    role="Risk Analyst",
    goal="Identify risks and opportunities for stocks",
    backstory="Macro and sector risk specialist",
    verbose=True
)


In [6]:
chat_advisor_agent = Agent(
    role="Investment Advisor",
    goal="Answer user questions about selected stocks",
    backstory="Friendly financial assistant explaining markets clearly",
    verbose=True
)


In [7]:
from crewai import Task

performance_task = Task(
    description="""
    Analyze stock price performance using provided price data.
    Identify trends, momentum, and notable movements.
    """,
    expected_output="Concise natural language performance summary",
    agent=stock_analyst_agent
)


In [8]:
risk_task = Task(
    description="""
    Analyze risks and opportunities considering:
    - Market conditions
    - Earnings
    - Competition
    - Macroeconomic trends
    """,
    expected_output="Risk and opportunity analysis per stock",
    agent=risk_analyst_agent
)


In [9]:
chat_task = Task(
    description="Answer user stock-related questions clearly and accurately",
    expected_output="Helpful and easy-to-understand answer",
    agent=chat_advisor_agent
)


In [10]:
from crewai import Crew

stock_crew = Crew(
    agents=[
        market_data_agent,
        stock_analyst_agent,
        risk_analyst_agent,
        chat_advisor_agent
    ],
    tasks=[performance_task, risk_task],
    verbose=True
)


In [11]:
def get_stock_summary_crewai(tickers, start, end):
    df = fetch_stock_data(tickers, start, end)

    context = {
        "tickers": tickers,
        "price_data": df.tail(5).to_string()
    }

    result = stock_crew.kickoff(inputs=context)
    return result


In [12]:
def get_risks_and_opportunities_crewai(tickers):
    context = {"tickers": tickers}
    result = stock_crew.kickoff(inputs=context)
    return result


In [13]:
def ask_ai_chat(query, tickers):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a financial assistant"},
            {"role": "user", "content": f"Stocks: {tickers}\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content


In [18]:
def init_ses_states():
    defaults = {
        "chat_history": [],
        "last_assets": [],
        "last_response": ""
    }
    for key, value in defaults.items():
        st.session_state.setdefault(key, value)

In [19]:
import streamlit as st 
import yfinance as yf
import pandas as pd
import openai
import os
from dotenv import load_dotenv
from htmlTemplates import css

def main():
    st.set_page_config(page_title="Stock Price AI Bot", page_icon=":chart:")
    st.write(css, unsafe_allow_html=True)
    init_ses_states()
    st.title("Stock Price AI Bot")
    st.caption("Visualizations & AI Insights for Stocks")

    with st.sidebar:
        with st.expander("Settings", expanded=True):
            asset_tickers = sorted(['DOW','NVDA','TSL','GOOGL','AMZN','AI','NIO','LCID','F','LYFY','AAPL', 'MSFT', 'BTC-USD', 'ETH-USD'])
            asset_dropdown = st.multiselect('Pick Assets:', asset_tickers)

            metric_tickers = ['Adj. Close', 'Relative Returns']
            metric_dropdown = st.selectbox("Metric", metric_tickers)

            viz_tickers = ['Line Chart', 'Area Chart']
            viz_dropdown = st.multiselect("Pick Charts:", viz_tickers)

            start = st.date_input('Start', value=pd.to_datetime('2023-01-01'))
            end = st.date_input('End', value=pd.to_datetime('today'))

    if len(asset_dropdown) == 0:
        st.warning("Please select at least one stock.")
    else:
        df = yf.download(asset_dropdown, start, end, auto_adjust=True)
    
        if df.empty:
            st.error(f"No data available for {asset_dropdown}. Please check the ticker symbols.")
            st.stop()
        
        st.write("Downloaded Data Preview:", df.head())

        # Select Adjusted Close price
        if 'Adj Close' in df.columns:
            df = df['Adj Close']
        elif 'Close' in df.columns:
            df = df['Close']
        else:
            st.error("Stock data does not contain 'Adj Close' or 'Close'. Check ticker symbols.")
            st.stop()

    if metric_dropdown == 'Relative Returns':
        df = relative_returns(df)
    
    if len(viz_dropdown) > 0:
        with st.expander(f"Data Visualizations for {metric_dropdown} of {asset_dropdown}", expanded=True):
            if "Line Chart" in viz_dropdown:
                st.subheader("Line Chart")
                st.line_chart(df)
            if "Area Chart" in viz_dropdown:
                st.subheader("Area Chart")
                st.area_chart(df)

    st.subheader("AI Stock Insights")

    # AI Stock Performance Summary
    if st.button("📊 Get AI Stock Performance Summary"):
        if not asset_dropdown:
            st.warning("⚠️ Please select at least one stock before requesting a summary.")
        else:
            summary = get_stock_summary(asset_dropdown, start, end)
            st.write("### AI Performance Summary:")
            st.write(summary)

    # AI Risks & Opportunities Analysis
    if st.button("💡 Get AI Risks & Opportunities Analysis"):
        if not asset_dropdown:
            st.warning("⚠️ Please select at least one stock before requesting risk & opportunity analysis.")
        else:
            risk_opportunities = get_risks_and_opportunities(asset_dropdown)
            st.write("### AI Risks & Opportunities:")
            st.write(risk_opportunities)

    # AI Chatbot for Stock Queries
    user_query = st.text_input("Ask AI about selected stocks:")
    if st.button("Ask AI"):
        if not asset_dropdown:
            st.warning("⚠️ Please select at least one stock before asking AI.")
        elif not user_query:
            st.warning("⚠️ Please enter a question before asking AI.")
        else:
            ai_response = ask_openai(user_query, asset_dropdown)
            st.write("### AI Response:")
            st.write(ai_response)

if __name__ == '__main__':
    main()

2026-01-25 16:40:26.116 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.125 Session state does not function when running a script without `streamlit run`
2026-01-25 16:40:26.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40:26.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 16:40